# Fermionic toric code — sign-head figures

Curated figures for the 3D fermionic toric-code sign-head study (branch `feat/fermionic-3d-signhead`).
Data: `results/fermionic_gate0/` (exact-diagonalization sign-fidelity ceilings), `results/fermionic_hx_ladder/`
and `results/fermionic_plane_L2/` (NQS trainings vs ED at L=2 OBC).

Figures are shown inline; `savefig` lines stay commented out (uncomment to commit a PNG).

## 1. Setup

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path("../../results")
FIGS = Path("../figs")

plt.rcParams.update({
    "figure.dpi": 120, "font.size": 11, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.3, "legend.frameon": False,
})

# Okabe–Ito, one colour per QEC head
HEAD_COLOR = {"plus": "#000000", "linear": "#0072B2", "vote": "#D55E00", "pt2": "#009E73"}
HEAD_LABEL = {"plus": "plus (no head)", "linear": "linear (MWPM)", "vote": "vote", "pt2": "pt2"}

## 2. Sign infidelity vs $h_x$ — 2×2×3 OBC (20 qubits)

$1-F_s$ is the $|\psi_{\rm ED}|^2$-weighted disagreement between a deterministic $\pm1$ sign head and the exact
ground-state sign (gauge $\psi(\text{all up})>0$): the fidelity ceiling of any positive trunk carrying that head.
Solid: $h_z=0$; dashed: $h_z=0.2$. Exact zeros are clipped to $10^{-17}$ for the log axis.

In [ ]:
gate0 = json.load(open(ROOT / "fermionic_gate0/2x2x3_OBC_gate0.json"))
pts = gate0["points"]
heads = ["plus", "linear", "vote", "pt2"]
FLOOR = 1e-17

fig, ax = plt.subplots(figsize=(6.4, 4.4))
for hz, ls in ((0.0, "-"), (0.2, "--")):
    rows = sorted([p for p in pts if abs(p["hz"] - hz) < 1e-9], key=lambda p: p["hx"])
    hx = np.array([p["hx"] for p in rows])
    for h in heads:
        y = np.array([max(p["one_minus_F_s"][h], FLOOR) for p in rows])
        ax.plot(hx, y, ls, marker="o", ms=4, color=HEAD_COLOR[h],
                label=HEAD_LABEL[h] if hz == 0.0 else None)
ax.set_yscale("log")
ax.set_xlabel(r"$h_x$")
ax.set_ylabel(r"$1-F_s$")
ax.set_title(r"sign infidelity, 2×2×3 OBC (solid $h_z=0$, dashed $h_z=0.2$)")
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1.0))
plt.show()
plt.savefig(FIGS / "fermionic_sign_infidelity_2x2x3_OBC.png", dpi=300, bbox_inches="tight")

## 3. In vivo on the magnetic line ($h_z=0$): relative energy error and infidelity vs $h_x$

L=2 OBC, 300 SR steps, cold start, exact evaluation against dense ED. Four trained tiers mapped onto the QEC heads:
**plus** = sign-blind approximately-symmetric trunk (`asymm`); **linear (MWPM) class** = frozen analytic head, $\kappa=0$
(`anaC_k0`; a fixed-recovery GF(2)-quadratic head, same family and same $h_x^2$ error scaling as the explicit linear decoder);
**vote** = vote head through the sign frame (`votesf`); **pt2** = pt2 head through the sign frame (`pt2sf`).
vote is drawn dashed with open markers because at L=2 OBC the vote never ties and the vote and pt2 heads are the same function, so their curves coincide. $\times$ marks a run stopped early by the divergence guard (banked at its last sane state).

In [ ]:
ladder = json.load(open(ROOT / "fermionic_hx_ladder/summary.json"))
TIER_HEAD = {"asymm": "plus", "anaC_k0": "linear", "pt2sf": "pt2", "votesf": "vote"}  # vote last: drawn on top of pt2
STYLE = {"plus": ("-", "o", None), "linear": ("-", "o", None), "pt2": ("-", "o", None), "vote": ("--", "o", "none")}
TIER_LABEL = {"asymm": "plus (no head)", "anaC_k0": "linear class (frozen anaC)", "votesf": "vote", "pt2sf": "pt2"}
FLOOR = 1e-16

fig, (axE, axF) = plt.subplots(1, 2, figsize=(11, 4.2))
for tier, head in TIER_HEAD.items():
    rows = sorted([r for r in ladder if r["tier"] == tier], key=lambda r: r["hx"])
    hx = np.array([r["hx"] for r in rows])
    rel = np.array([max(r["rel"], FLOOR) for r in rows])
    inf = np.array([max(1.0 - r["fidelity"], FLOOR) for r in rows])
    div = np.array([bool(r["diverged"]) for r in rows])
    c = HEAD_COLOR[head]
    ls, mk, mfc = STYLE[head]
    kw = dict(ls=ls, marker=mk, ms=6 if mfc == "none" else 4, mfc=mfc, color=c, label=TIER_LABEL[tier])
    axE.plot(hx, rel, **kw)
    axF.plot(hx, inf, **kw)
    for ax, y in ((axE, rel), (axF, inf)):
        ax.plot(hx[div], y[div], "x", ms=9, mew=1.8, color=c)
for ax, yl, title in ((axE, r"$|E-E_0|/|E_0|$", "relative energy error"),
                      (axF, r"$1-F$", "trained infidelity")):
    ax.set_yscale("log"); ax.set_xlabel(r"$h_x$"); ax.set_ylabel(yl); ax.set_title(title)
axF.legend(loc="upper left", bbox_to_anchor=(1.02, 1.0))
fig.suptitle(r"fermionic L=2 OBC, magnetic line ($h_z=0$)", y=1.02)
plt.show()
plt.savefig(FIGS / "fermionic_hx_ladder_relerr_infidelity.png", dpi=300, bbox_inches="tight")

## 4. The $(h_x, h_z)$ plane at L=2 OBC: heatmaps

4×4 grid $\{0, 0.2, 0.5, 1.0\}^2$, 300 SR steps, cold start, exact evaluation against dense ED. Columns are the trained
arms mapped onto the QEC heads (plus = sign-blind trunk; linear class = frozen anaC, $\kappa=0$; pt2 through the sign frame
with a positive real trunk and with a complex trunk). First figure: relative energy error (Blues). Second figure: trained
infidelity $1-F$ (Purples). One shared log colour scale per figure; $^{\times}$ marks a guard-terminated run.

In [ ]:
import sys
from matplotlib.colors import LogNorm
sys.path.insert(0, "../scripts")
import plane_summary

GRID = [0.0, 0.2, 0.5, 1.0]
ARMS = [("asymm", "plus (no head)"), ("anaC_k0", "linear class (frozen anaC)"),
        ("pt2sf", "pt2, real trunk"), ("pt2sfc", "pt2, complex trunk")]
rows = plane_summary.build(ROOT / "fermionic_plane_L2")

def grid_of(arm, key):
    g = np.full((len(GRID), len(GRID)), np.nan); flag = np.zeros_like(g, dtype=bool)
    for r in rows:
        if r["arm"] != arm: continue
        i, j = GRID.index(r["hz"]), GRID.index(r["hx"])
        g[i, j] = r["rel"] if key == "rel" else 1.0 - r["fidelity"]
        flag[i, j] = bool(r["diverged"])
    return g, flag

def plane_figure(key, cmap, label, title):
    """One 1x4 row (one panel per arm), shared log colour scale, annotated cells (same layout as fermionic_figs.py)."""
    grids = [grid_of(arm, key) for arm, _ in ARMS]
    vals = np.concatenate([g[np.isfinite(g)] for g, _ in grids])
    norm = LogNorm(vmin=max(vals.min(), 1e-14), vmax=vals.max())
    fig, axes = plt.subplots(1, len(ARMS), figsize=(3.5 * len(ARMS) + 1.3, 4.3), sharey=True, constrained_layout=True)
    for ax, (arm, name), (g, flag) in zip(axes, ARMS, grids):
        im = ax.imshow(np.clip(g, norm.vmin, None), origin="lower", cmap=cmap, norm=norm, interpolation="nearest")
        ax.set_xticks(range(len(GRID)), GRID); ax.set_yticks(range(len(GRID)), GRID)
        ax.set_xlabel(r"$h_x$"); ax.set_title(name, fontsize=10); ax.grid(False)
        for i in range(len(GRID)):
            for j in range(len(GRID)):
                if not np.isfinite(g[i, j]): continue
                dark = norm(max(g[i, j], norm.vmin)) > 0.6
                ax.text(j, i, f"{g[i, j]:.1e}" + (r"$^{\times}$" if flag[i, j] else ""), ha="center", va="center",
                        fontsize=8, color="white" if dark else "black")
    axes[0].set_ylabel(r"$h_z$")
    fig.colorbar(im, ax=axes, shrink=0.9, pad=0.02, label=label)
    fig.suptitle(title)
    return fig

plane_figure("rel", "Blues", r"$|E-E_0|/|E_0|$", "fermionic L=2 OBC plane: relative energy error")
plt.show()
plt.savefig(FIGS / "fermionic_plane_L2_relerr.png", dpi=300, bbox_inches="tight")

plane_figure("inf", "Purples", r"$1-F$", "fermionic L=2 OBC plane: trained infidelity")
plt.show()
plt.savefig(FIGS / "fermionic_plane_L2_infidelity.png", dpi=300, bbox_inches="tight")
